# Software-as-a-Graph (SaG) — JSS Journal Paper Reproducibility Suite
### *Heterogeneous Graph Learning for Pre-Deployment Reliability and Dependability Analysis of Complex Distributed Systems*
**Journal of Systems and Software (JSS) — Special Issue VSI:AI4MSS**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/onuralpyigit/SoftwareAsAGraph/blob/main/notebooks/train_gnn_colab.ipynb)

This notebook provides a cloud-ready, GPU-accelerated environment to train all GNN models and reproduce the empirical results, tables, and figures for the JSS submission.

---

### Hardware Accelerator Setup (Colab GPU)
1. In the Colab menu, go to **Runtime** > **Change runtime type**.
2. Under **Hardware accelerator**, select **T4 GPU** (free tier) or **A100 / V100** (Colab Pro).
3. Click **Save**.

## 1. Workspace Setup
Clones the repository from GitHub for code + committed data (`data/scenarios/`). Two gitignored cache directories are required and can only be rebuilt against a live Neo4j instance, which Colab doesn't provide — so both are pulled in separately from Google Drive as pre-built tarballs instead of being part of the clone:
- `output/loso_cache/` — required by `main_table.py`/`loso_all_variants.py`/`kfold_all_variants.py`. Must contain all 12 paper scenarios (`atm_system`, `av_system`, `iot_smart_city_system`, `financial_trading_system`, `healthcare_system`, `hub_and_spoke_system`, `microservices_system`, `enterprise_system`, `telecom_ran_system`, `industrial_scada_system`, `realtime_gaming_system`, `logistics_fleet_system`) — LOSO (Table 8) and k-fold (Table 9) discover scenarios purely from what's physically present under it, with no flag to catch a partial cache.
- `output/realworld_cache/` — required by `realworld_zeroshot.py` (Table 11). Must contain all 5 real-world systems (`realworld_autoware_ros2`, `realworld_cloud_microservices`, `realworld_trainticket`, `realworld_homeassistant`, `realworld_edgex`). Kept as a **separate** directory from `loso_cache` — `discover_scenarios` treats every folder it finds as a LOSO fold, so merging the two would silently change the 12-fold corpus behind every published LOSO number.

The cell below checks coverage for both and warns if anything is missing.

**One-time local pre-flight** (run on a machine with both caches already built, e.g. via `make -f reproduce/Makefile cache` for the synthetic one and `scripts/populate_loso_cache.sh` pointed at `CACHE_DIR=output/realworld_cache` for the real-world one — see `reproduce/realworld_zeroshot.py`'s docstring):
```bash
tar -czf output/loso_cache.tar.gz -C output loso_cache
tar -czf output/realworld_cache.tar.gz -C output realworld_cache
```
Upload both to `My Drive/SaG/` once. The cell below then mounts Drive only to fetch those two files.

In [ ]:
# --- Primary path: clone code from GitHub, pull loso_cache from Drive ---
import os
from pathlib import Path

REPO_DIR = "/content/SoftwareAsAGraph"
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/onuralpyigit/SoftwareAsAGraph.git {REPO_DIR}
else:
    # Pull latest changes if the repo already exists
    !cd {REPO_DIR} && git pull


%cd {REPO_DIR}
%set_env PYTHONPATH=.

from google.colab import drive
drive.mount('/content/drive')

def _extract_cache(tarball_name, dest_dir):
    tarball = f"/content/drive/MyDrive/SaG/{tarball_name}"
    if os.path.exists(tarball):
        !mkdir -p output
        !tar -xzf {tarball} -C output/
        print(f"✓ {dest_dir} extracted from Drive.")
    else:
        print(f"⚠️ {tarball} not found — cells depending on {dest_dir} will fail "
              "until it's built locally and uploaded there.")

_extract_cache("loso_cache.tar.gz", "output/loso_cache")
_extract_cache("realworld_cache.tar.gz", "output/realworld_cache")

# --- Scenario coverage check ---
# loso_all_variants.py / kfold_all_variants.py take no --scenarios flag: they
# discover scenarios purely from whatever folders are physically present under
# --cache-dir. A tarball built before a domain was added (or built with only the
# 7 in-distribution scenarios) will silently run an 8- or 7-fold LOSO instead of
# the paper's 12-fold LOSO (JSS Table 8), with no error anywhere downstream — so
# check coverage here, before spending GPU time on a run that under-covers it.
REQUIRED_LOSO_SCENARIOS = [
    "atm_system", "av_system", "iot_smart_city_system", "financial_trading_system",
    "healthcare_system", "hub_and_spoke_system", "microservices_system",
    "enterprise_system", "telecom_ran_system", "industrial_scada_system",
    "realtime_gaming_system", "logistics_fleet_system",
]
REQUIRED_REALWORLD_SYSTEMS = [
    "realworld_autoware_ros2", "realworld_cloud_microservices",
    "realworld_trainticket", "realworld_homeassistant", "realworld_edgex",
]

def _check_coverage(cache_dir, required, label):
    cache_dir = Path(cache_dir)
    present = sorted(p.name for p in cache_dir.iterdir() if p.is_dir()) if cache_dir.exists() else []
    missing = [s for s in required if s not in present]
    if missing:
        print(f"⚠️ {cache_dir} is missing {len(missing)}/{len(required)} required {label}: {missing}")
    else:
        print(f"✓ All {len(required)} {label} present in {cache_dir}: {present}")

_check_coverage("output/loso_cache", REQUIRED_LOSO_SCENARIOS, "LOSO scenarios")
_check_coverage("output/realworld_cache", REQUIRED_REALWORLD_SYSTEMS, "real-world systems")

# --- Alternative: mount the whole project from Drive instead of cloning ---
# from google.colab import drive
# drive.mount('/content/drive')
# # Option: entire project zipped in Drive:
# # !unzip -q /content/drive/MyDrive/SoftwareAsAGraph.zip -d /content/SoftwareAsAGraph
# REPO_DIR = "/content/SoftwareAsAGraph"
# %cd {REPO_DIR}
# %set_env PYTHONPATH=.


## 2. Hardware Verification & Dependencies
Verify GPU availability and install PyTorch Geometric along with the repository package.

In [ ]:
# Inspect GPU hardware
!nvidia-smi

import torch
print("PyTorch Version:", torch.__version__)
print("CUDA Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU Device Name:", torch.cuda.get_device_name(0))
    print("Device Count:", torch.cuda.device_count())
else:
    print("⚠️ GPU not detected. Please enable GPU in Runtime > Change runtime type.")

In [ ]:
# Install PyTorch Geometric, pytest, and SaaG core package
!pip install -q torch-geometric pytest
!pip install -q -e .
print("✓ Dependencies installed successfully.")

## 3. Block 0: QoS Pipeline Audit (Go/No-Go Gate)
Runs the W1 QoS pipeline audit gate (`tests/test_qos_pipeline_audit.py` and `tests/test_baselines.py`) to verify topological formulation integrity and baseline correctness before training.

In [ ]:
!pytest tests/test_qos_pipeline_audit.py tests/test_baselines.py -v --tb=short -q

## 4. Smoke-Test (Fast Sanity Check)
Runs a lightweight training sweep on a **3-scenario subset** (`atm_system`, `av_system`, `iot_smart_city_system`) — deliberately smaller than the paper's 7-scenario Table 6 set, just to verify end-to-end execution before running the full 300-epoch matrix. Takes ~1–2 minutes on GPU.

In [ ]:
!python reproduce/main_table.py \
    --scenarios atm_system av_system iot_smart_city_system \
    --seeds 42 123 \
    --epochs 50 \
    --device auto \
    --output results/smoke_main_table.json \
    --resume

## 5. Experiment 1: In-Distribution Evaluation (JSS Table 6 & Table 7)
Trains all 12 scenarios × 6 model variants × 5 random seeds = 360 evaluation cells — the manuscript's original 7 in-distribution domains plus the 4 domains added for LOSO (`telecom_ran_system`, `industrial_scada_system`, `realtime_gaming_system`, `logistics_fleet_system`) and `atm_system`, matching the full 12-scenario coverage now in `output/loso_cache`:
`atm_system`, `av_system`, `iot_smart_city_system`, `financial_trading_system`, `healthcare_system`, `hub_and_spoke_system`, `microservices_system`, `enterprise_system`, `telecom_ran_system`, `industrial_scada_system`, `realtime_gaming_system`, `logistics_fleet_system`.

> **Note:** this is a broader scenario set than the manuscript's current Table 6/7 (7 scenarios, 210 cells) — reconcile against `docs/research/jss/latex/sections/sec7_results.tex` before treating this run's numbers as a drop-in replacement for the published table.

- **Structural Baselines**: `Topo`, `Topo-QoS`
- **Homogeneous GNNs**: `GAT`, `GAT-QoS`
- **Heterogeneous GNNs**: `HGT` (masked QoS ablation), `HGT-QoS` (proposed model)

*Resilience*: Uses `--resume` so if the Colab session disconnects, re-running skips completed runs.

In [ ]:
# Train full in-distribution matrix — all 12 scenarios (see note above)
!python reproduce/main_table.py \
    --scenarios atm_system av_system iot_smart_city_system financial_trading_system \
                healthcare_system hub_and_spoke_system microservices_system enterprise_system \
                telecom_ran_system industrial_scada_system realtime_gaming_system logistics_fleet_system \
    --output results/main_table.json \
    --seeds 42 123 456 789 2024 \
    --epochs 300 \
    --device auto \
    --resume

In [ ]:
# Render LaTeX and Markdown tables (results/table3_main_results.tex & .md)
!python reproduce/render_table.py \
    --table3 results/main_table.json \
    --output-dir results

# Display rendered markdown table inline
from IPython.display import display, Markdown
if os.path.exists("results/table3_main_results.md"):
    with open("results/table3_main_results.md") as f:
        display(Markdown(f.read()))

## 6. Experiment 2: Inductive Cross-Domain Generalization (LOSO, JSS Table 8)
Evaluates cross-domain generalization via Leave-One-Scenario-Out (LOSO) cross-validation and computes Wilcoxon signed-rank tests. `loso_all_variants.py` takes no `--scenarios` flag — it folds over whatever scenario folders are physically present under `--cache-dir output/loso_cache`, so this is a 12-fold LOSO (the paper's 7 in-distribution scenarios plus `atm_system`, `telecom_ran_system`, `industrial_scada_system`, `realtime_gaming_system`, `logistics_fleet_system`) **only if** the coverage check in step 1 reported all 12 present.

In [ ]:
# Train LOSO across all variants
!python reproduce/loso_all_variants.py \
    --cache-dir output/loso_cache \
    --epochs 300 \
    --device auto \
    --output results/loso_all_variants.json \
    --resume

# Render Table 8 LaTeX output
!python reproduce/render_table.py \
    --table4 results/loso_all_variants.json \
    --output-dir results

# Calculate statistical significance & p-values
!python reproduce/loso_significance.py \
    --input results/loso_all_variants.json \
    --output results/loso_significance.json

In [ ]:
# Display summary of LOSO significance
import json
if os.path.exists("results/loso_significance.json"):
    with open("results/loso_significance.json") as f:
        sig = json.load(f)
    print("=== LOSO Wilcoxon Significance Results ===")
    print(json.dumps(sig, indent=2))

## 7. Experiment 3: In-Domain Per-Domain K-Fold Evaluation (JSS Table 9)
Trains 5 variants across $k$ folds per domain scenario (opt-in validation protocol). Like LOSO, `kfold_all_variants.py` discovers scenarios from `--cache-dir output/loso_cache` rather than a `--scenarios` flag, so it also depends on the step 1 coverage check reporting all 12 scenarios present.

In [ ]:
# Optional: run per-domain k-fold cross-validation
!python reproduce/kfold_all_variants.py \
    --cache-dir output/loso_cache \
    --epochs 300 \
    --device auto \
    --output results/kfold_all_variants.json \
    --resume

!python reproduce/render_table.py \
    --table-kfold results/kfold_all_variants.json \
    --output-dir results

## 8. Experiment 4: Real-World Zero-Shot Transfer (JSS Table 11)
Trains HGT-QoS once on all 12 synthetic scenarios (2 message-passing layers, 150 epochs, 5 seeds, within-graph rank normalization) and scores it zero-shot against $I^*(v)$ on the five open-source real-world systems — none of them contribute training gradients or are used for checkpoint selection. Requires `output/realworld_cache` (pulled from Drive in step 1). Defaults below match the paper's methodology exactly; no flags needed. Writes `results/realworld_zeroshot.json` and prints its own summary table.

In [ ]:
# Zero-shot transfer of HGT-QoS to five real-world systems
!python reproduce/realworld_zeroshot.py --device auto

## 9. Case Study & Figures (Attention Subgraphs & JSS Figures 3–5)
Generates the publication figures:
- **Figure S2 / Figure 5**: ATM Case Study HGT Attention Subgraph.
- **Figure 3**: Results-at-a-glance (LOSO Spearman $\rho$, F1@K, Oracle agreement).
- **Figure 4**: Stratified per-node-type $\rho$.

In [ ]:
# Extract attention weights for ATM scenario and render subgraph
!python reproduce/extract_attention.py \
    --scenario atm_system \
    --device auto \
    --output-dir output/atm_case_study

!python reproduce/render_attention_subgraph.py \
    --input output/atm_case_study/attention_weights.json \
    --output output/atm_case_study/attention_subgraph

# Render results-at-a-glance figure
!python reproduce/render_results_figure.py
!python reproduce/render_stratified_figure.py --source auto --output results/figure4_stratified_rho

In [ ]:
# Display rendered figures
from IPython.display import Image, display
from pathlib import Path

figures = [
    Path("output/atm_case_study/attention_subgraph.png"),
    Path("docs/research/jss/latex/figures/Figure_3.png"),
    Path("results/figure4_stratified_rho.png")
]

for fig in figures:
    if fig.exists():
        print(f"\nFigure: {fig}")
        display(Image(filename=str(fig)))

## 10. Backup Checkpoints & Results to Google Drive
Persists trained model checkpoints (`output/gnn_checkpoints/`) and output tables/figures (`results/`) to Google Drive.

In [ ]:
from datetime import datetime

backup_dir = f"/content/drive/MyDrive/SaG_JSS_Results_{datetime.now().strftime('%Y%m%d_%H%M')}"
os.makedirs(backup_dir, exist_ok=True)

!cp -r results {backup_dir}/
if os.path.exists("output/gnn_checkpoints"):
    !cp -r output/gnn_checkpoints {backup_dir}/
if os.path.exists("output/atm_case_study"):
    !cp -r output/atm_case_study {backup_dir}/

print(f"✓ Experimental artifacts backed up to: {backup_dir}")